In [8]:
# Ensure required packages are installed in the notebook environment
import os
import shutil
import ssl
from doctr.models import ocr_predictor
from doctr.io import DocumentFile
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
#another message

In [9]:
# --- CONFIGURATION ---
DB_DIR = "./chroma_db" # may need to reconfigure if updates are applied to ./reference-health-files
PDF_DATA_DIR = "./reference-health-files" # all PDFs needed for sourcing will be stored here
IMAGE_PATHS = ["john_doe_patient_statement.jpg", "john_doe_eob.jpg"] # medical bills to analyze

In [10]:
# --- INITIALIZE MODELS ---
# Using all-MiniLM-L6-v2 for semantic vectors as per your notes
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# LLM
llm = Ollama(model="llama3.2", temperature=0, num_ctx=4096) # make sure to run ollama pull llama3.2

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7031.12it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# creation of vector that stores comprehensive information based on a folder of PDFs

def build_knowledge_base():
    """Builds the vector database from PDFs in the reference folder."""
    if not os.path.exists(PDF_DATA_DIR):
        os.makedirs(PDF_DATA_DIR)
        print(f"Created {PDF_DATA_DIR}. Add your PDFs there and run again.")
        return None

    print(f"Loading PDFs from {PDF_DATA_DIR}...")
    loader = DirectoryLoader(PDF_DATA_DIR, glob="./*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    
    # Recursive chunking to keep semantic meaning intact
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)
    
    print(f"Vectorizing {len(chunks)} chunks into ChromaDB...")
    # Delete old DB if it exists to prevent 'readonly' errors
    if os.path.exists(DB_DIR):
        shutil.rmtree(DB_DIR)
        
    vector_db = Chroma.from_documents(
        documents=chunks, 
        embedding=embeddings, 
        persist_directory=DB_DIR
    )
    print("Knowledge base built successfully.")
    return vector_db

In [11]:
# this function will turn 2 JPGs into a structured text that is ready to be analyzed

def extract_text_from_images(image_list):
    """Processes multiple JPGs and merges their text."""
    combined_text = ""
    
    # Initialize the model once outside the loop for speed
    model = ocr_predictor(det_arch='db_resnet50', reco_arch='crnn_vgg16_bn', pretrained=True)
    
    for idx, img_path in enumerate(image_list):
        if not os.path.exists(img_path):
            print(f"Skipping: {img_path} not found.")
            continue
            
        print(f"Performing OCR on Page {idx + 1}: {img_path}...")
        doc = DocumentFile.from_images([img_path])
        result = model(doc)
        
        # Add a marker so the LLM knows where pages start/end
        combined_text += f"\n--- START OF PAGE {idx + 1} ---\n"
        
        for page in result.pages:
            for block in page.blocks:
                for line in block.lines:
                    line_text = " ".join([word.value for word in line.words])
                    combined_text += line_text + "\n"
                    
    return combined_text

In [12]:
def analyze_bill_with_rag(bill_text):
    """Uses RAG to explain the OCR text using the PDF knowledge base."""
    vector_db = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    
    template = """
    ### SYSTEM INSTRUCTION ###
    You are a Medical Billing Specialist, but don't state this at the beginning of the response. 
    Compare the Statement to the EOB. Use the provided context to analyze the bills.
    Be factual, but also speak with less medical jargon and easier words.
    Additionally, make a recommendation based on context and current state of the bill of what steps a patient should take
    for financing their bill or if they should consider specific insurance coverage options.
    Speak to the user with no assumption of how good their English is and be as simple, but straight to the point as possible.
    If the info isn't in the context, say you aren't sure.
    Remember that you are addressing the user directly. You are not a chatbot and you will just summarize the bills.

    ### RESEARCH CONTEXT ###
    {context}

    ### PATIENT BILL ###
    {question}

    ### ANALYSIS REPORT ###
    """
    PROMPT = PromptTemplate(template=template, input_variables=["context", "question"])
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
        chain_type_kwargs={"prompt": PROMPT}
    )
    
    print("Generating analysis...")
    response = qa_chain.invoke(bill_text)
    return response["result"]

In [13]:
# EXECUTION

if __name__ == "__main__":
    ssl._create_default_https_context = ssl._create_unverified_context

    if not os.path.exists(DB_DIR):
        build_knowledge_base()
    
    # Define your list of images
    bill_images = ["john_doe_patient_statement.jpg", "john_doe_eob.jpg"]
    
    # Step A: Get all text from all pages
    full_bill_content = extract_text_from_images(bill_images)
    
    if full_bill_content.strip():
        # Step B: Run one single analysis on the total content
        analysis = analyze_bill_with_rag(full_bill_content)
        
        print("\n" + "="*40)
        print("MEDICAL BILL AUDIT REPORT (MULTI-PAGE)")
        print("="*40)
        print(analysis)
    else:
        print("No text was extracted. Please check your image paths.")

Performing OCR on Page 1: john_doe_patient_statement.jpg...
Performing OCR on Page 2: john_doe_eob.jpg...
Generating analysis...

MEDICAL BILL AUDIT REPORT (MULTI-PAGE)
Let's go through the bills and statements you received.

First, I want to explain what we're looking at here. The Statement is from NewYork-Presbyterian Hospital, which shows how much you owe for your medical care. The Explanation of Benefits (EOB) is from Aetna Health Insurance, which explains how much they paid for the same services.

Let's look at the bills:

* You had an emergency visit on March 15th, and the hospital charged $1,450. However, Aetna only paid $640, leaving a balance of $810.
* You also had a CT scan on the same day, which cost $2,100. But again, Aetna only paid $960.

The problem is that the charges for both services exceed what's allowed by your insurance plan. This means you have a high deductible and copay for these services.

Based on this information, I would recommend paying the remaining balan